# Task 5 - Detecting Data Leakage and Evaluation Issues

This notebook reviews the provided pipeline, explains the methodological problems, and runs a corrected version.

In [1]:
import sys
from pathlib import Path

project_root = Path.cwd().resolve()
if not (project_root / "results").exists():
    project_root = project_root.parents[1]
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from results.codex.task5.generated_code import review_pipeline, run_corrected_pipeline, plot_metrics, plot_confusion_matrix, corrected_pipeline_code


## Review The Existing Pipeline

This step identifies leakage, preprocessing errors, and evaluation weaknesses in the provided script.

In [2]:
issues_df = review_pipeline()
issues_df

=== Reviewed Pipeline ===
task/task5_pipeline.py

=== Identified Issues ===
                                                 issue                                                                                                                             why_problematic                                                                                                                                correction
             Preprocessing before the train/test split                  The scaler is fit on the full dataset before splitting, so information from the test set leaks into the training workflow.                Fit preprocessing only on the training data by placing it inside an sklearn Pipeline or ColumnTransformer after the split.
Mixed data types are passed directly to StandardScaler The dataset contains categorical string columns such as Gender and Class, so the current code will fail instead of producing a valid model. Handle categorical and numeric features separately, for example

                                                 issue                                                                                                                             why_problematic                                                                                                                                correction
             Preprocessing before the train/test split                  The scaler is fit on the full dataset before splitting, so information from the test set leaks into the training workflow.                Fit preprocessing only on the training data by placing it inside an sklearn Pipeline or ColumnTransformer after the split.
Mixed data types are passed directly to StandardScaler The dataset contains categorical string columns such as Gender and Class, so the current code will fail instead of producing a valid model. Handle categorical and numeric features separately, for example with one-hot encoding for categoricals and scaling only numeric features.
 

## Run The Corrected Pipeline

This version drops the identifier, uses a stratified split, and keeps imputation, encoding, and scaling inside a train-only sklearn pipeline.

In [3]:
metrics_df, corrected_pipeline, X_test, y_test = run_corrected_pipeline()
plot_metrics(metrics_df, fig_id=1)
plot_confusion_matrix(corrected_pipeline, X_test, y_test, fig_id=2)
metrics_df


Loading dataset from: data/processed/codex_clean.csv
Dataset shape: (129880, 24)

=== Corrected Pipeline Metrics ===
                        model  accuracy       f1  roc_auc
Corrected Logistic Regression  0.874346 0.852431  0.92769
Saved figure 1 to: figures/codex/fig_01_task5_corrected_pipeline_metrics.png
Figure(700x400)
Saved figure 2 to: figures/codex/fig_02_task5_corrected_confusion_matrix.png
Figure(500x400)


                        model  accuracy       f1  roc_auc
Corrected Logistic Regression  0.874346 0.852431  0.92769

## Corrected Pipeline Template

This final step prints a compact corrected example that follows sound machine learning practice.

In [4]:
print(corrected_pipeline_code())

df = pd.read_csv("data/processed/codex_clean.csv")
df = df.drop(columns=["id"])

X = df.drop(columns=["satisfaction"])
y = (df["satisfaction"] == "satisfied").astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

categorical_features = X.select_dtypes(include=["object", "category"]).columns.tolist()
numeric_features = X.select_dtypes(include=["number"]).columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            Pipeline(
                steps=[
                    ("imputer", SimpleImputer(strategy="most_frequent")),
                    ("encoder", OneHotEncoder(handle_unknown="ignore")),
                ]
            ),
            categorical_features,
        ),
        (
            "numeric",
            Pipeline(
                steps=[
                    ("imputer", SimpleImputer(strategy="median")),
                    ("scaler", StandardScaler()),
